# micrograd_plus — Interactive Demo

This notebook walks through the key features of **micrograd_plus**:
1. Tensor basics and automatic differentiation
2. Functional operations (`ops`)
3. Building neural networks with `nn`
4. Training with optimizers
5. XOR problem end-to-end

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from micrograd import Tensor, ops
from micrograd.nn import Sequential, Linear, ReLU, Sigmoid, Dropout
from micrograd.nn.loss import MSELoss, BCELoss, CrossEntropyLoss
from micrograd.optim import SGD, Adam

print('micrograd_plus loaded successfully!')

## 1. Tensor Basics

In [ ]:
# Create tensors
a = Tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
b = Tensor([[5.0, 6.0], [7.0, 8.0]], requires_grad=True)

print('a:', a)
print('b:', b)
print('shape:', a.shape)
print('dtype:', a.dtype)

In [ ]:
# Basic arithmetic
c = a + b
d = a * b
e = a @ b.T  # matrix multiply

print('a + b =', c.data)
print('a * b =', d.data)
print('a @ b.T =', e.data)

## 2. Automatic Differentiation

In [ ]:
# Simple gradient computation
x = Tensor(np.array([1.0, 2.0, 3.0]), requires_grad=True)

# f(x) = sum(x^2)
# df/dx = 2x
loss = (x ** 2).sum()
loss.backward()

print('x     =', x.data)
print('df/dx =', x.grad)  # Should be [2, 4, 6]
print('Expected: [2, 4, 6]')

In [ ]:
# Chain rule example: d/dx sigmoid(x^2)
x = Tensor(np.array([0.5, 1.0, -0.5]), requires_grad=True)

out = ops.sigmoid(x ** 2).sum()
out.backward()

print('x         =', x.data)
print('grad      =', x.grad)

# Numerical verification
eps = 1e-5
num_grad = np.zeros(3)
for i in range(3):
    x_plus = x.data.copy(); x_plus[i] += eps
    x_minus = x.data.copy(); x_minus[i] -= eps
    s_plus = 1/(1+np.exp(-x_plus**2))
    s_minus = 1/(1+np.exp(-x_minus**2))
    num_grad[i] = (s_plus.sum() - s_minus.sum()) / (2*eps)

print('numerical =', num_grad)
print('match:', np.allclose(x.grad, num_grad, atol=1e-5))

## 3. Functional Operations

In [ ]:
x_vals = np.linspace(-3, 3, 100)
x_t = Tensor(x_vals)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Activation Functions', fontsize=14)

activations = [
    ('ReLU', ops.relu),
    ('Sigmoid', ops.sigmoid),
    ('Tanh', ops.tanh),
    ('exp', ops.exp),
    ('log (clipped)', lambda t: ops.log(Tensor(np.abs(t.data) + 0.1))),
    ('sqrt', lambda t: ops.sqrt(Tensor(np.abs(t.data)))),
]

for ax, (name, fn) in zip(axes.flat, activations):
    y = fn(x_t).data
    ax.plot(x_vals, y)
    ax.set_title(name)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

## 4. Building Neural Networks

In [ ]:
# Define a simple MLP
model = Sequential(
    Linear(4, 16),
    ReLU(),
    Dropout(0.2),
    Linear(16, 8),
    ReLU(),
    Linear(8, 3)
)

print(model)
print()

# Count parameters
n_params = sum(p.data.size for p in model.parameters())
print(f'Total parameters: {n_params}')

In [ ]:
# Forward pass
x = Tensor(np.random.randn(5, 4))  # batch of 5
out = model(x)
print('Input shape:', x.shape)
print('Output shape:', out.shape)

## 5. Loss Functions

In [ ]:
# MSE Loss
pred = Tensor(np.array([1.5, 2.3, 0.8]), requires_grad=True)
target = Tensor(np.array([1.0, 2.0, 1.0]))

mse = MSELoss()(pred, target)
print(f'MSE Loss: {mse.item():.6f}')

mse.backward()
print(f'Gradient: {pred.grad}')

In [ ]:
# Cross Entropy Loss
logits = Tensor(np.array([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3]]), requires_grad=True)
targets = np.array([0, 1])  # class indices

ce_loss = CrossEntropyLoss()(logits, targets)
print(f'CrossEntropy Loss: {ce_loss.item():.6f}')

ce_loss.backward()
print(f'Gradient shape: {logits.grad.shape}')
print(f'Gradient:\n{logits.grad}')

## 6. Optimizers

In [ ]:
# Compare SGD and Adam on a simple quadratic
# f(x) = x^2, minimum at x=0

def optimize(optimizer_class, **kwargs):
    np.random.seed(42)
    x = Tensor(np.array([5.0]), requires_grad=True)
    opt = optimizer_class([x], **kwargs)
    losses = []
    for _ in range(100):
        loss = (x ** 2).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(float(x.data[0]))
    return losses

sgd_path = optimize(SGD, lr=0.1)
sgd_mom_path = optimize(SGD, lr=0.1, momentum=0.9)
adam_path = optimize(Adam, lr=0.5)

plt.figure(figsize=(10, 4))
plt.plot(sgd_path, label='SGD lr=0.1')
plt.plot(sgd_mom_path, label='SGD momentum=0.9')
plt.plot(adam_path, label='Adam lr=0.5')
plt.xlabel('Step')
plt.ylabel('Parameter value (should → 0)')
plt.title('Optimizer comparison: minimizing f(x) = x²')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. XOR Problem End-to-End

In [ ]:
np.random.seed(42)

# XOR dataset
X = Tensor([[0,0],[0,1],[1,0],[1,1]], requires_grad=False)
y = Tensor([[0],[1],[1],[0]], requires_grad=False)

# Model
model = Sequential(
    Linear(2, 16),
    ReLU(),
    Linear(16, 1),
    Sigmoid()
)

optimizer = Adam(list(model.parameters()), lr=0.01)
criterion = BCELoss()

losses = []
for epoch in range(3000):
    pred = model(X)
    loss = criterion(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('XOR Training Loss')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

print(f'Final loss: {losses[-1]:.6f}')

In [ ]:
# Visualize the decision boundary
model.eval()

xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 100), np.linspace(-0.5, 1.5, 100))
grid = Tensor(np.c_[xx.ravel(), yy.ravel()])
Z = model(grid).data.reshape(xx.shape)

plt.figure(figsize=(7, 6))
plt.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.8)
plt.colorbar(label='P(y=1)')
plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

colors = ['blue', 'red']
for (xi, yi_val), label in zip(X.data, y.data):
    plt.scatter(xi[0], xi[1], c=colors[int(label[0])], s=200, zorder=5,
                edgecolors='black', linewidths=2)

plt.xlabel('x1')
plt.ylabel('x2')
plt.title('XOR Decision Boundary\n(red=class 1, blue=class 0)')
plt.show()

# Print predictions
preds = model(X)
print('\nFinal predictions:')
for xi, pi, ti in zip(X.data, preds.data, y.data):
    pred_class = int(pi[0] > 0.5)
    print(f'  {xi.astype(int)} → {pi[0]:.4f} → class {pred_class} (true: {int(ti[0])})')

## 8. Gradient Checking Demo

In [ ]:
def numerical_gradient(f, x, eps=1e-5):
    grad = np.zeros_like(x.data)
    it = np.nditer(x.data, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        orig = x.data[idx]
        x.data[idx] = orig + eps
        f_plus = f(x).item()
        x.data[idx] = orig - eps
        f_minus = f(x).item()
        x.data[idx] = orig
        grad[idx] = (f_plus - f_minus) / (2 * eps)
        it.iternext()
    return grad

print('Gradient check results:')
print('-' * 50)

tests = [
    ('sum(x^2)',     lambda x: (x**2).sum()),
    ('relu(x)',      lambda x: ops.relu(x).sum()),
    ('sigmoid(x)',   lambda x: ops.sigmoid(x).sum()),
    ('tanh(x)',      lambda x: ops.tanh(x).sum()),
    ('softmax(x)',   lambda x: ops.softmax(x).sum()),
]

for name, f in tests:
    x = Tensor(np.random.randn(3, 4) * 0.5, requires_grad=True)
    out = f(x)
    out.backward()
    analytical = x.grad.copy()
    
    x2 = Tensor(x.data.copy(), requires_grad=True)
    numerical = numerical_gradient(f, x2)
    
    max_err = np.abs(analytical - numerical).max()
    status = 'PASS' if max_err < 1e-4 else 'FAIL'
    print(f'  [{status}] {name:20s}  max_err={max_err:.2e}')